# Core scaling: MESSI, SOFA, and TRIE

This notebook reads the archived `MESSI_QUERY_*.csv` and `MESSI_SETTINGS_*.csv` files produced by `run_core_scaling_experiment.sh`. It reports macro-averaged query latency over datasets for 16, 32, and 64 query workers, and plots the scaling curves. The index worker count is read from the settings files but is not used as the x-axis.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The first path is the layout created when this notebook is run from notebooks/.
ROOT_CANDIDATES = [Path('trie_logs/core_scaling'), Path('notebooks/trie_logs/core_scaling')]
LOG_ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), None)
if LOG_ROOT is None:
    raise FileNotFoundError('Could not find trie_logs/core_scaling (run from the repository or notebooks directory).')

QUERY_COLUMNS = ['querying time', 'total']

def _settings_map(path):
    # The command row has many comma-separated fields, so parse only the first comma.
    values = {}
    for line in path.read_text(errors='replace').splitlines()[1:]:
        if ',' not in line:
            continue
        key, value = line.split(',', 1)
        values[key.strip().strip(chr(34)).lower()] = value.strip().strip(chr(34))
    return values

def _setting_file(query_file):
    stamp = query_file.stem.replace('MESSI_QUERY_', '')
    candidate = query_file.parent.parent / 'settings' / f'MESSI_SETTINGS_{stamp}.csv'
    if candidate.exists():
        return candidate
    matches = list(query_file.parents[3].glob(f'**/settings/MESSI_SETTINGS_{stamp}.csv'))
    return matches[0] if matches else None

def _method_name(system, settings):
    function_type = settings.get('function type', '')
    histogram_type = settings.get('histogram type', '')
    prefix = {'trie': 'TRIE', 'sofa': 'SOFA', 'messi': 'MESSI'}.get(system, system.upper())
    if function_type == '3' or system == 'messi':
        return f'{prefix} / SAX'
    family = 'SPARTAN' if function_type == '5' or system == 'trie' else 'SFA'
    variant = {'1': 'depth', '2': 'width'}.get(histogram_type, histogram_type or 'unknown')
    return f'{prefix} / {family}-{variant}'

def load_core_scaling(root=LOG_ROOT):
    rows = []
    for query_file in sorted(root.rglob('MESSI_QUERY_*.csv')):
        settings_file = _setting_file(query_file)
        if settings_file is None:
            continue
        settings = _settings_map(settings_file)
        try:
            cores = int(settings['threads'])
        except (KeyError, ValueError):
            # Archived results also encode the query worker count in the parent directory.
            try:
                cores = int(query_file.parents[1].name)
            except ValueError:
                continue
        if cores not in (16, 32, 64):
            continue
        system = next((part for part in query_file.parts if part in {'trie', 'sofa', 'messi'}), 'unknown')
        frame = pd.read_csv(query_file)
        if len(frame) > 1:
            frame = frame.iloc[:-1].copy()  # final row is the aggregate written by MESSI
        time_col = next((c for c in QUERY_COLUMNS if c in frame.columns), None)
        if time_col is None:
            continue
        # MESSI writes querying time in microseconds; report milliseconds.
        latency_ms = pd.to_numeric(frame[time_col], errors='coerce').dropna() / 1000.0
        if latency_ms.empty:
            continue
        dataset = settings.get('dataset', query_file.parents[2].name)
        dataset = Path(dataset).stem or query_file.parents[2].name
        rows.append({
            'method': _method_name(system, settings),
            'system': system,
            'dataset': dataset,
            'cores': cores,
            'index_threads': int(settings.get('index threads', 0) or 0),
            'query_ms': float(latency_ms.mean()),
            'queries': int(latency_ms.size),
            'source': str(query_file),
        })
    result = pd.DataFrame(rows)
    if result.empty:
        raise RuntimeError(f'No core-scaling query files found below {root}')
    return result

runs = load_core_scaling()
# Equal-weight every dataset, so a large dataset does not dominate the summary.
summary = (runs.groupby(['method', 'cores'], as_index=False)
           .agg(datasets=('dataset', 'nunique'),
                mean_query_ms=('query_ms', 'mean'),
                sd_query_ms=('query_ms', 'std'),
                index_threads=('index_threads', 'median')))
summary['sd_query_ms'] = summary['sd_query_ms'].fillna(0.0)
summary = summary.sort_values(['method', 'cores'])
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
print(f'Loaded {len(runs)} run files from {LOG_ROOT}')
print('\nPer-method core-scaling table (macro-average over datasets; query time in ms):')
display(summary)

wide = summary.pivot(index='method', columns='cores', values='mean_query_ms').reindex(columns=[16, 32, 64])
print('\nCompact table (mean query ms):')
display(wide)

fig, ax = plt.subplots(figsize=(8.5, 5.0))
for method, group in summary.groupby('method', sort=True):
    group = group.sort_values('cores')
    ax.errorbar(group['cores'], group['mean_query_ms'], yerr=group['sd_query_ms'],
                marker='o', linewidth=2, capsize=3, label=method)
ax.set_xticks([16, 32, 64])
ax.set_xlabel('Query workers / cores')
ax.set_ylabel('Mean query time (ms)')
ax.set_title('Core scaling across datasets')
ax.grid(axis='y', alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

# Optional machine-readable output next to the copied logs.
# summary.to_csv(LOG_ROOT / 'core_scaling_summary.csv', index=False)